# Lakehouse Agent - Optional Cleanup

This notebook cleans up **all** AWS resources created by notebooks **01–09**, for the **selected IdP** (`IDP_PROVIDER`).

**⚠️ WARNING: This will delete all resources created during deployment!**

Each deployment step has a dedicated cleanup script. This notebook runs them in **reverse deployment order**, with `## [COGNITO]` / `## [OKTA]` guards on the IdP-specific steps (the flag is read once, below).

**Resources torn down:**
- Identity Provider — **[COGNITO]** User Pool + domain + post-auth Lambda + login-audit table, or **[OKTA]** apps + auth server + groups + users
- **GW1** claims gateway + REQUEST/RESPONSE interceptors + DynamoDB tenant-role map
- **GW2** notes gateway + target + role + OBO/M2M credential providers (+ agent-IAM revert)
- **[COGNITO]** notes REQUEST interceptor (Lambda + role + log group)
- **4a** lakehouse MCP runtime + **4b** OpenSearch MCP runtime (IAM/ECR/CodeBuild each)
- **AOSS** collection `lakehouse-claim-notes` + its encryption/network/data policies
- **S3 Tables** + Lake Formation registration (**pre-existing/shared LF admins are preserved** — fork B17)
- **IAM** tenant roles + LF data-access role
- **S3** bucket (optional) + all **SSM** parameters under `/app/lakehouse-agent/`

**Prerequisites:**
- AWS credentials configured
- Python 3.10 or later
- **[OKTA]** only: `OKTA_ORG_URL` + `OKTA_API_TOKEN` set in `.env` (required by `cleanup_okta.py`)

In [ ]:
# AWS Initialization
from utils.notebook_init import init_aws
from utils.idp_config import get_idp_provider
import subprocess

session, region, account_id = init_aws()
ssm_client = session.client("ssm", region_name=region)

# Read the IdP flag ONCE; every guard below branches on this variable
# (no per-cell SSM re-read).
IDP_PROVIDER = get_idp_provider(ssm_client)

print("✅ Ready to clean up")
print(f"   Account ID: {account_id}")
print(f"   Region: {region}")
print(f"   IdP Provider: {IDP_PROVIDER}")

## Step 1: Delete Lakehouse Agent Runtime

Deletes the agent runtime, IAM role, ECR repository, and CodeBuild project. Applies to both IdPs (the agent is IdP-agnostic).

In [ ]:
# No capture_output: stdout/stderr stream live so the multi-minute delete shows progress.
result = subprocess.run(
    ["python", "cleanup_agent.py"],
    cwd="deployment/6-lakehouse-agent",
)
if result.returncode != 0:
    print(f"⚠️  Errors above (returncode {result.returncode})")

## Step 2: Delete Notes Gateway (GW2), OpenSearch OBO/M2M & Notes Interceptor

`06_cleanup_obo_gateway.py` (**both IdPs**) tears down the GW2 notes gateway + target + IAM role, **both** OAuth2 credential providers (Okta OBO `lakehouse-obo-okta-provider` + Cognito M2M `lakehouse-notes-cognito-oauth-provider`, each safe-if-absent), the **AOSS** collection `lakehouse-claim-notes` + its policies, and reverts the agent IAM OBO patch. The Okta-only bits are `⏭️` no-ops on Cognito and vice-versa.

> ⏳ **The AOSS collection delete blocks until the collection is fully removed (~10 min).** This is expected — let the cell run.

Then, on **## [COGNITO]** only, `interceptor-notes/cleanup.sh` removes the notes REQUEST interceptor (Lambda `lakehouse-notes-interceptor` + role + log group + SSM). Okta has no notes interceptor, so this step is skipped there.

In [ ]:
# No capture_output: stdout/stderr stream live so the multi-minute delete shows progress.
result = subprocess.run(
    ["python", "06_cleanup_obo_gateway.py"],
    cwd="deployment/5b-obo-gateway-setup",
)
if result.returncode != 0:
    print(f"⚠️  Errors above (returncode {result.returncode})")

In [ ]:
## [COGNITO] — Notes REQUEST interceptor teardown (skipped on Okta)
if IDP_PROVIDER == "cognito":
    # No capture_output: stdout/stderr stream live so the multi-minute delete shows progress.
    result = subprocess.run(
        ["bash", "cleanup.sh"],
        cwd="deployment/5a-gateway-setup/interceptor-notes",
    )
    if result.returncode != 0:
        print(f"⚠️  Errors above (returncode {result.returncode})")
else:
    print("⏭️  [OKTA] No Cognito notes interceptor to delete — skipping")

## Step 3: Delete Claims Gateway (GW1) & Interceptors

Deletes the GW1 claims gateway + targets, the REQUEST + RESPONSE interceptor Lambdas and their IAM role, the OAuth2 providers, the **DynamoDB tenant-role mapping table**, and the gateway role. Applies to both IdPs (GW1 topology is identical on both paths).

In [ ]:
# No capture_output: stdout/stderr stream live so the multi-minute delete shows progress.
result = subprocess.run(
    ["python", "cleanup_gateway.py"],
    cwd="deployment/5a-gateway-setup",
)
if result.returncode != 0:
    print(f"⚠️  Errors above (returncode {result.returncode})")

## Step 4: Delete MCP Server Runtimes (lakehouse 4a + OpenSearch 4b)

Deletes both MCP server runtimes and their IAM roles, ECR repositories, and CodeBuild projects. Both are IdP-agnostic (deployed on both paths).

In [ ]:
# No capture_output: stdout/stderr stream live so the multi-minute delete shows progress.
result = subprocess.run(
    ["python", "cleanup_runtime.py"],
    cwd="deployment/4a-mcp-lakehouse-server",
)
if result.returncode != 0:
    print(f"⚠️  Errors above (returncode {result.returncode})")

In [ ]:
# No capture_output: stdout/stderr stream live so the multi-minute delete shows progress.
result = subprocess.run(
    ["python", "cleanup_runtime.py"],
    cwd="deployment/4b-mcp-opensearch-server",
)
if result.returncode != 0:
    print(f"⚠️  Errors above (returncode {result.returncode})")

## Step 5: Delete S3 Tables & Lake Formation

Deletes the S3 Tables bucket, namespace, tables, federated catalog, and **deregisters** the S3 Tables resource from Lake Formation. 

> 🔒 **B17:** this step does **not** remove any Lake Formation *administrators* — pre-existing/shared LF admins are preserved (no `put_data_lake_settings` on teardown; `deregister_resource` only).

In [ ]:
# No capture_output: stdout/stderr stream live so the multi-minute delete shows progress.
result = subprocess.run(
    ["python", "cleanup_s3tables.py"],
    cwd="deployment/3-s3tables-setup",
)
if result.returncode != 0:
    print(f"⚠️  Errors above (returncode {result.returncode})")

## Step 6: Delete IAM Tenant Roles

Deletes the policyholders, adjusters, and administrators tenant roles plus the Lake Formation data-access role. Applies to both IdPs.

In [ ]:
# No capture_output: stdout/stderr stream live so the multi-minute delete shows progress.
result = subprocess.run(
    ["python", "cleanup_iam_roles.py"],
    cwd="deployment/2-lakehouse-tenant-roles-setup",
)
if result.returncode != 0:
    print(f"⚠️  Errors above (returncode {result.returncode})")

## Step 7: Delete Identity Provider

Branches on `IDP_PROVIDER`:
- **## [COGNITO]** — `cleanup_cognito.py`: User Pool + domain, post-auth Lambda + role, login-audit DynamoDB table.
- **## [OKTA]** — `cleanup_okta.py`: OIDC app + OBO exchange app, custom auth server, groups, test users, `okta-*` SSM. **Requires `OKTA_ORG_URL` + `OKTA_API_TOKEN` in `.env`** (the script exits 1 without them), so it is **only** ever run on the Okta path.

In [ ]:
## [COGNITO] — Cognito teardown (skipped on Okta)
if IDP_PROVIDER == "cognito":
    # No capture_output: stdout/stderr stream live so the multi-minute delete shows progress.
    result = subprocess.run(
        ["python", "cleanup_cognito.py"],
        cwd="deployment/1-cognito-setup",
    )
    if result.returncode != 0:
        print(f"⚠️  Errors above (returncode {result.returncode})")
else:
    print("⏭️  [OKTA] Skipping Cognito teardown")

In [ ]:
## [OKTA] — Okta teardown (skipped on Cognito; needs OKTA_ORG_URL + OKTA_API_TOKEN)
if IDP_PROVIDER == "okta":
    # No capture_output: stdout/stderr stream live so the multi-minute delete shows progress.
    result = subprocess.run(
        ["python", "cleanup_okta.py"],
        cwd="deployment/1-okta-setup",
    )
    if result.returncode != 0:
        print(f"⚠️  Errors above (returncode {result.returncode})")
else:
    print("⏭️  [COGNITO] Skipping Okta teardown")

## Step 8: Delete S3 Bucket (Optional)

**⚠️ This permanently deletes all data in the S3 bucket!**

Disabled by default. Set `DELETE_S3_BUCKET = True` to enable.

In [ ]:
DELETE_S3_BUCKET = False  # Change to True to permanently delete the S3 bucket + all its data

if not DELETE_S3_BUCKET:
    print("⏭️  S3 bucket deletion is DISABLED")
    print("   Set DELETE_S3_BUCKET = True to enable")
else:
    s3_client = session.client("s3", region_name=region)
    try:
        bucket_name = ssm_client.get_parameter(Name="/app/lakehouse-agent/s3-bucket-name")["Parameter"]["Value"]
        print(f"Deleting all objects in: {bucket_name}")
        paginator = s3_client.get_paginator("list_objects_v2")
        for page in paginator.paginate(Bucket=bucket_name):
            if "Contents" in page:
                objects = [{"Key": obj["Key"]} for obj in page["Contents"]]
                s3_client.delete_objects(Bucket=bucket_name, Delete={"Objects": objects})
        s3_client.delete_bucket(Bucket=bucket_name)
        print(f"✅ Deleted S3 bucket: {bucket_name}")
    except Exception as e:
        print(f"❌ Error: {e}")

## Step 9: Delete All SSM Parameters

Bulk-deletes every parameter under `/app/lakehouse-agent/` — catches per-user sub keys (`cognito-user-*-sub` / `okta-user-*-sub`), `notes-gateway-*`, `notes-interceptor-lambda-arn`, `idp-provider`, and everything else the scripts above may have left with `--keep-ssm`.

In [ ]:
print("🗑️  Deleting all SSM parameters...\n")
try:
    paginator = ssm_client.get_paginator("describe_parameters")
    params_to_delete = []
    for page in paginator.paginate(
        ParameterFilters=[{"Key": "Name", "Option": "BeginsWith", "Values": ["/app/lakehouse-agent/"]}]
    ):
        params_to_delete.extend([p["Name"] for p in page["Parameters"]])

    if params_to_delete:
        for i in range(0, len(params_to_delete), 10):
            batch = params_to_delete[i : i + 10]
            ssm_client.delete_parameters(Names=batch)
            for p in batch:
                print(f"  ✅ Deleted: {p}")
        print(f"\n✅ Deleted {len(params_to_delete)} SSM parameters")
    else:
        print("⏭️  No SSM parameters found")
except Exception as e:
    print(f"❌ Error: {e}")

## Summary

In [ ]:
print("=" * 60)
print("🎉 CLEANUP COMPLETE")
print("=" * 60)
print()
print(f"IdP Provider: {IDP_PROVIDER}")
print()
print("Shared resources cleaned up (both IdPs):")
print("  • Lakehouse Agent Runtime + IAM role + ECR")
print("  • GW2 notes gateway + target + role + OBO/M2M providers (+ agent-IAM revert)")
print("  • GW1 claims gateway + REQUEST/RESPONSE interceptors + DynamoDB tenant-role map")
print("  • MCP runtimes: 4a lakehouse + 4b OpenSearch (IAM/ECR/CodeBuild each)")
print("  • AOSS collection lakehouse-claim-notes + policies")
print("  • S3 Tables + Lake Formation deregistration (LF admins PRESERVED — B17)")
print("  • IAM tenant roles + LF data-access role")
print("  • S3 bucket (if enabled) + all SSM parameters")
print()
if IDP_PROVIDER == "cognito":
    print("Cognito-specific resources cleaned up:")
    print("  • Cognito User Pool + domain + post-auth Lambda + login-audit table")
    print("  • Notes REQUEST interceptor (Lambda + role + log group)")
else:
    print("Okta-specific resources cleaned up:")
    print("  • Okta OIDC app + OBO exchange app + auth server + groups + users")
print()
print("Manual cleanup (if needed):")
print("  • CloudWatch Log Groups: /aws/bedrock-agentcore/runtime/*")
print("  • CloudWatch Log Groups: /aws/lambda/lakehouse-*")